# Payload Validation Guide

`TelemetryPayload` is the single source of truth for every piece of telemetry
that flows through the EWIS toolkit.  Whether you are feeding data into the
analysis engine, computing grid-stress metrics, or exporting results to an
external API, the payload schema defines exactly what is accepted, what is
optional, and what constraints each field must satisfy.  Treating it as a
strict contract — rather than a loose dictionary — is the fastest way to
eliminate an entire class of data-quality bugs before they reach production.

This notebook walks through every field, demonstrates the built-in Pydantic
validation rules, shows how to surface and interpret validation errors, and
offers practical advice for integrating payload validation into your own
pipelines.  By the end you should be able to construct payloads confidently,
understand why a given input was rejected, and decide which optional fields
to populate for the analysis you need.

## Required Fields

Every `TelemetryPayload` **must** include the six fields listed below.
Omitting any of them — or supplying a value that violates its constraint —
will raise a `ValidationError` at construction time.

| Field | Type | Constraint | Description |
|---|---|---|---|
| `datacenter_id` | `str` | min length 1 (non-empty) | Unique identifier for the datacenter |
| `region` | `str` | min length 1 (non-empty) | Geographic region code (e.g. `us-east-1`) |
| `timestamp_utc` | `str` | min length 1 (non-empty) | ISO-8601 UTC timestamp of the reading |
| `power_mw` | `float` | ≥ 0.0 | Total facility power draw in megawatts |
| `it_load_mw` | `float` | ≥ 0.0 | IT equipment load in megawatts |
| `pue` | `float` | ≥ 1.0 | Power Usage Effectiveness (total power / IT load) |

In [ ]:
from ewis.core.payloads import TelemetryPayload

# Minimal valid payload — only the six required fields
minimal = TelemetryPayload(
    datacenter_id="dc-west-01",
    region="us-west-2",
    timestamp_utc="2024-07-01T12:00:00Z",
    power_mw=42.0,
    it_load_mw=35.0,
    pue=1.2,
)

print(minimal.to_dict())

## Optional Fields

The remaining fields default to `None` when omitted.  They enrich the
payload with grid, economic, environmental, and workload context — but the
core pipeline will still run without them (metrics that depend on missing
fields gracefully return `None` instead of crashing).

| Field | Type | Constraint | Description |
|---|---|---|---|
| `base_grid_load_mw` | `float \| None` | ≥ 0.0 | Background grid demand excluding the datacenter (MW) |
| `grid_capacity_mw` | `float \| None` | ≥ 0.0 | Total available grid capacity (MW) |
| `price_usd_per_mwh` | `float \| None` | ≥ 0.0 | Real-time electricity spot price (USD/MWh) |
| `carbon_intensity_kgco2_per_mwh` | `float \| None` | ≥ 0.0 | Grid carbon intensity (kg CO₂/MWh) |
| `ambient_temp_c` | `float \| None` | *(none)* | Outside air temperature in °C (can be negative) |
| `humidity_pct` | `float \| None` | 0.0 – 100.0 | Relative humidity percentage |
| `wind_m_s` | `float \| None` | ≥ 0.0 | Wind speed in metres per second |
| `precip_mm` | `float \| None` | ≥ 0.0 | Precipitation in millimetres |
| `workload` | `Dict[str, Any] \| None` | *(none)* | Free-form workload metadata (tokens, energy_kwh, etc.) |

In [ ]:
# Fully populated payload — every field supplied
full = TelemetryPayload(
    datacenter_id="dc-east-02",
    region="us-east-1",
    timestamp_utc="2024-07-01T14:30:00Z",
    power_mw=100.0,
    it_load_mw=80.0,
    pue=1.25,
    base_grid_load_mw=500.0,
    grid_capacity_mw=800.0,
    price_usd_per_mwh=45.50,
    carbon_intensity_kgco2_per_mwh=0.42,
    ambient_temp_c=33.5,
    humidity_pct=72.0,
    wind_m_s=3.1,
    precip_mm=0.0,
    workload={
        "model": "gpt-4-turbo",
        "tokens": 1_500_000,
        "energy_kwh": 120.0,
        "batch_id": "run-20240701-001",
    },
)

print(full.to_dict())

## JSON Schema Export

Because `TelemetryPayload` is a Pydantic v2 model, you can call
`model_json_schema()` to produce a standards-compliant JSON Schema document.
This is especially useful when you need to:

* Share the schema with a front-end or partner team that does not use Python.
* Register the schema in an API gateway or event-bus registry.
* Auto-generate documentation or client-side validators.

In [ ]:
import json

print(json.dumps(TelemetryPayload.model_json_schema(), indent=2))

## Validation Rules in Action

Pydantic validates every field the moment you instantiate a model.  If any
constraint is violated, a `ValidationError` is raised that lists **all**
failing fields at once (not just the first one).  The sections below
demonstrate each built-in rule so you know exactly what to expect when bad
data arrives.

### Rule 1: Required fields cannot be empty

String fields (`datacenter_id`, `region`, `timestamp_utc`) enforce a
minimum length of 1.  Passing an empty string is treated the same as a
missing value.

In [ ]:
from pydantic import ValidationError

try:
    TelemetryPayload(
        datacenter_id="",          # ← empty string violates min_length=1
        region="us-west-2",
        timestamp_utc="2024-07-01T12:00:00Z",
        power_mw=42.0,
        it_load_mw=35.0,
        pue=1.2,
    )
except ValidationError as exc:
    print("ValidationError caught!")
    print(exc)

### Rule 2: power_mw and it_load_mw must be non-negative

Both fields carry a `ge=0.0` constraint — negative energy values are
physically meaningless for a datacenter power reading.

In [ ]:
try:
    TelemetryPayload(
        datacenter_id="dc-west-01",
        region="us-west-2",
        timestamp_utc="2024-07-01T12:00:00Z",
        power_mw=-5,               # ← negative value violates ge=0.0
        it_load_mw=35.0,
        pue=1.2,
    )
except ValidationError as exc:
    print("ValidationError caught!")
    print(exc)

### Rule 3: PUE must be ≥ 1.0

Power Usage Effectiveness is defined as *total facility power ÷ IT
equipment power*.  Because total power always includes IT power plus
overhead (cooling, lighting, etc.), PUE is **always ≥ 1.0** by definition.
A value below 1.0 indicates a sensor or calculation error.

In [ ]:
try:
    TelemetryPayload(
        datacenter_id="dc-west-01",
        region="us-west-2",
        timestamp_utc="2024-07-01T12:00:00Z",
        power_mw=42.0,
        it_load_mw=35.0,
        pue=0.8,                   # ← below 1.0 is physically impossible
    )
except ValidationError as exc:
    print("ValidationError caught!")
    print(exc)

# PUE refresher:
# PUE = Total Facility Power / IT Equipment Power
# Since overhead ≥ 0, Total ≥ IT, therefore PUE ≥ 1.0 always.

### Rule 4: it_load_mw must not exceed power_mw

A custom `field_validator` enforces that IT load never exceeds total
facility power.  This is a cross-field check: Pydantic validates
`power_mw` first (because it appears earlier in the model), then passes
its value to the `it_load_mw` validator via `info.data`.

In [ ]:
try:
    TelemetryPayload(
        datacenter_id="dc-west-01",
        region="us-west-2",
        timestamp_utc="2024-07-01T12:00:00Z",
        power_mw=30.0,
        it_load_mw=50.0,           # ← 50 > 30 → fails cross-field check
        pue=1.2,
    )
except ValidationError as exc:
    print("ValidationError caught!")
    print(exc)

### Rule 5: humidity_pct must be 0–100

Relative humidity is expressed as a percentage.  The field carries both a
lower bound (`ge=0.0`) and an upper bound (`le=100.0`).  Values outside
this range indicate faulty sensor data or unit-conversion errors.

In [ ]:
try:
    TelemetryPayload(
        datacenter_id="dc-west-01",
        region="us-west-2",
        timestamp_utc="2024-07-01T12:00:00Z",
        power_mw=42.0,
        it_load_mw=35.0,
        pue=1.2,
        humidity_pct=120.0,        # ← above 100% violates le=100.0
    )
except ValidationError as exc:
    print("ValidationError caught!")
    print(exc)

## Working with Partial Data

In many real-world scenarios you will not have values for every optional
field — perhaps the grid operator API is down, or weather telemetry has not
arrived yet.  The EWIS toolkit is designed for **graceful degradation**:
metrics that depend on missing fields return `value=None` together with an
explanatory `notes` string, rather than raising an exception.  This lets
your pipeline continue processing the fields it *does* have.

In [ ]:
from ewis.metrics.core import grid_stress_index

# Build a payload with only the required fields — no grid or weather data
partial = TelemetryPayload(
    datacenter_id="dc-west-01",
    region="us-west-2",
    timestamp_utc="2024-07-01T12:00:00Z",
    power_mw=42.0,
    it_load_mw=35.0,
    pue=1.2,
)

# The payload itself is perfectly valid
print("Payload dict:", partial.to_dict())
print()

# grid_stress_index needs grid_capacity_mw and base_grid_load_mw.
# Because they are missing it returns value=None with an explanatory note.
result = grid_stress_index(partial.to_dict())
print("GSI value:", result["value"])
print("GSI notes:", result["notes"])

## Best Practices

1. **Always validate before calling `engine.run()`.**  Constructing a
   `TelemetryPayload` is cheap; debugging a half-finished pipeline run is
   not.  Validate early so errors surface at the data-ingestion boundary.

2. **Use `to_dict()` for serialization.**  The helper calls
   `model_dump(mode="python", exclude_none=True)`, which strips `None`
   values and returns native Python types — ready for JSON encoding or
   insertion into a database.

3. **Add optional fields incrementally.**  Start with the six required
   fields, confirm your pipeline works end-to-end, then enrich the payload
   one data source at a time.  This makes it easy to isolate which new
   field introduced a data-quality issue.

4. **Use `model_json_schema()` for API documentation.**  Embed the
   generated JSON Schema in your OpenAPI spec or event-bus registry so
   every consumer knows the exact contract without reading Python source.

5. **Log validation errors in production.**  Catch `ValidationError`,
   serialize it with `exc.json()`, and write it to your structured-logging
   system.  This gives you a queryable record of every malformed payload
   your service received, making it easy to trace upstream data issues.